In [1]:
import os, glob, math, random
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt
import torch.nn.functional as F

charset_file = "/content/kashmiri_charset.txt"
with open(charset_file, encoding="utf-8") as f:
    CHARS = f.read().strip()

CHAR2IDX = {c: i+1 for i, c in enumerate(CHARS)}
IDX2CHAR = {i: c for c, i in CHAR2IDX.items()}

def text_to_labels(text):
    return [CHAR2IDX.get(c, 0) for c in text]

def labels_to_text(labels):
    return ''.join([IDX2CHAR.get(i, '') for i in labels])

def get_all_fonts_image_label_paths(base_dirs):
    if isinstance(base_dirs, str):
        base_dirs = [base_dirs]

    all_images, all_labels = [], []

    for base_dir in base_dirs:
        for font_name in os.listdir(base_dir):
            font_dir = os.path.join(base_dir, font_name)
            if not os.path.isdir(font_dir):
                continue

            for split in ["train", "val"]:
                split_dir = os.path.join(font_dir, split)
                if not os.path.isdir(split_dir):
                    continue

                images = glob.glob(os.path.join(split_dir, "*.png"))
                labels = [img.replace(".png", ".txt") for img in images]
                all_images.extend(images)
                all_labels.extend(labels)

    combined = list(zip(all_images, all_labels))
    random.shuffle(combined)
    all_images, all_labels = zip(*combined)

    return list(all_images), list(all_labels)

class OCRDataset(Dataset):
    def __init__(self, image_paths, label_paths, transform=None):
        self.samples = list(zip(image_paths, label_paths))
        self.samples.sort()
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, txt_path = self.samples[idx]

        image = Image.open(img_path).convert("L")

        if self.transform:
            image = self.transform(image)

        with open(txt_path, encoding="utf-8") as f:
            label = f.read().strip()

        label_idx = text_to_labels(label)

        return image, torch.tensor(label_idx, dtype=torch.long), len(label_idx)

def collate_fn(batch):
    images, labels, label_lens = zip(*batch)

    images = torch.stack(images)
    label_lens = torch.tensor(label_lens, dtype=torch.long)

    labels_flat = torch.cat(labels) if labels else torch.tensor([], dtype=torch.long)

    max_len = max([len(l) for l in labels])

    padded_labels = torch.zeros(
        len(labels),
        max_len,
        dtype=torch.long
    )

    for i, l in enumerate(labels):
        padded_labels[i, :len(l)] = l

    return images, labels_flat, padded_labels, label_lens

class STN(nn.Module):
    def __init__(self, img_h, img_w):
        super().__init__()

        self.localization = nn.Sequential(
            nn.Conv2d(1, 8, 7, padding=3),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(8, 16, 5, padding=2),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        fc_input = 16 * (img_h//4) * (img_w//4)

        self.fc_loc = nn.Sequential(
            nn.Linear(fc_input, 32),
            nn.ReLU(),
            nn.Linear(32, 6)
        )

        self.fc_loc[2].weight.data.zero_()

        self.fc_loc[2].bias.data.copy_(
            torch.tensor(
                [1,0,0,0,1,0],
                dtype=torch.float
            )
        )

    def forward(self, x):
        xs = self.localization(x)
        xs = xs.view(xs.size(0), -1)

        theta = self.fc_loc(xs).view(-1,2,3)

        grid = nn.functional.affine_grid(
            theta,
            x.size(),
            align_corners=False
        )

        x = nn.functional.grid_sample(
            x,
            grid,
            align_corners=False
        )

        return x

class ResidualBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, stride, 1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(),
            nn.Conv2d(out_ch, out_ch, 3, 1, 1),
            nn.BatchNorm2d(out_ch)
        )

        self.relu = nn.ReLU()

        self.downsample = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 1, stride),
            nn.BatchNorm2d(out_ch)
        ) if in_ch != out_ch or stride != 1 else nn.Identity()

    def forward(self, x):
        out = self.conv(x)
        out += self.downsample(x)
        return self.relu(out)

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()

        pe = torch.zeros(max_len, d_model)

        position = torch.arange(
            0,
            max_len
        ).unsqueeze(1).float()

        div_term = torch.exp(
            torch.arange(
                0,
                d_model,
                2
            ).float()
            * (-math.log(10000.0)/d_model)
        )

        pe[:,0::2] = torch.sin(
            position * div_term
        )

        pe[:,1::2] = torch.cos(
            position * div_term
        )

        pe = pe.unsqueeze(0)

        self.register_buffer("pe", pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return x

class SmallTransformerEncoder(nn.Module):
    def __init__(
        self,
        feature_dim,
        nhead=4,
        num_layers=2,
        dropout=0.1
    ):
        super().__init__()

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=feature_dim,
            nhead=nhead,
            dim_feedforward=feature_dim*2,
            dropout=dropout,
            batch_first=True
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        self.pos_encoder = PositionalEncoding(
            feature_dim
        )

    def forward(self, x):
        x = self.pos_encoder(x)
        return self.encoder(x)

class AttentionDecoder(nn.Module):
    def __init__(
        self,
        feature_dim,
        hidden_dim,
        n_classes,
        dropout=0.1
    ):
        super().__init__()

        self.hidden_dim = hidden_dim

        self.embedding = nn.Embedding(
            n_classes,
            hidden_dim
        )

        self.lstm = nn.LSTM(
            hidden_dim + feature_dim,
            hidden_dim,
            batch_first=True
        )

        self.attn_fc = nn.Linear(
            hidden_dim + feature_dim,
            1
        )

        self.out_fc = nn.Linear(
            hidden_dim,
            n_classes
        )

        self.dropout = nn.Dropout(dropout)

    def forward(
        self,
        encoder_outputs,
        targets=None,
        teacher_forcing_ratio=0.5
    ):
        batch_size, seq_len, feature_dim = encoder_outputs.size()
        device = encoder_outputs.device

        max_len = (
            targets.size(1)
            if targets is not None
            else seq_len
        )

        h = torch.zeros(
            1,
            batch_size,
            self.hidden_dim,
            device=device
        )

        c = torch.zeros(
            1,
            batch_size,
            self.hidden_dim,
            device=device
        )

        inputs = torch.zeros(
            batch_size,
            1,
            self.hidden_dim,
            device=device
        )

        outputs = []

        for t in range(max_len):
            attn_weights = F.softmax(
                self.attn_fc(
                    torch.cat(
                        [
                            inputs.squeeze(1).unsqueeze(1).repeat(
                                1,
                                seq_len,
                                1
                            ),
                            encoder_outputs
                        ],
                        dim=2
                    )
                ),
                dim=1
            )

            context = torch.sum(
                attn_weights * encoder_outputs,
                dim=1,
                keepdim=True
            )

            lstm_input = torch.cat(
                [inputs, context],
                dim=2
            )

            out, (h,c) = self.lstm(
                lstm_input,
                (h,c)
            )

            out = self.dropout(out)

            pred = self.out_fc(out)

            outputs.append(pred)

            if (
                targets is not None
                and torch.rand(1).item() < teacher_forcing_ratio
            ):
                idx = targets[:,t].unsqueeze(1)
                inputs = self.embedding(idx)
            else:
                idx = pred.argmax(-1)
                inputs = self.embedding(idx)

        outputs = torch.cat(
            outputs,
            dim=1
        )

        return outputs

class CRNN_STN_Transformer_Attn(nn.Module):
    def __init__(
        self,
        img_h=64,
        n_channels=1,
        n_classes=len(CHAR2IDX)+1,
        dropout=0.2,
        hidden_dim=256
    ):
        super().__init__()

        self.n_classes = n_classes

        self.stn = STN(
            img_h,
            256
        )

        self.cnn = nn.Sequential(
            ResidualBlock(
                n_channels,
                64
            ),
            nn.MaxPool2d(
                2,
                2
            ),
            ResidualBlock(
                64,
                128
            ),
            nn.MaxPool2d(
                2,
                2
            ),
            ResidualBlock(
                128,
                256
            ),
            nn.MaxPool2d(
                (2,1)
            ),
            ResidualBlock(
                256,
                256
            ),
            nn.MaxPool2d(
                (2,1)
            ),
            nn.Dropout(dropout)
        )

        self.h_out = img_h // 16

        feature_dim = 256 * self.h_out

        self.transformer = SmallTransformerEncoder(
            feature_dim,
            nhead=4,
            num_layers=2,
            dropout=0.1
        )

        self.fc_ctc = nn.Linear(
            feature_dim,
            n_classes
        )

        self.attn_decoder = AttentionDecoder(
            feature_dim,
            hidden_dim,
            n_classes,
            dropout
        )

    def forward(
        self,
        x,
        targets=None,
        teacher_forcing_ratio=0.5
    ):
        x = self.stn(x)

        x = self.cnn(x)

        b,c,h,w = x.size()

        x = x.permute(
            0,
            3,
            1,
            2
        ).contiguous().view(
            b,
            w,
            c*h
        )

        enc_out = self.transformer(x)

        ctc_out = self.fc_ctc(
            enc_out
        ).log_softmax(2)

        attn_out = self.attn_decoder(
            enc_out,
            targets,
            teacher_forcing_ratio
        )

        return ctc_out, attn_out

def attn_greedy_decoder(
    attn_out,
    blank=0
):
    preds = attn_out.argmax(-1)

    pred_texts = []

    for pred in preds:
        seq = [
            p.item()
            for p in pred
            if p.item() != blank
        ]

        pred_texts.append(seq)

    return pred_texts

def char_accuracy(
    preds,
    labels,
    label_lens
):
    total_chars, correct_chars = 0, 0

    for i, l_len in enumerate(label_lens):
        true_seq = labels[
            i,
            :l_len
        ].cpu().tolist()

        pred_seq = (
            preds[i][:l_len]
            if len(preds[i]) >= l_len
            else preds[i] + [0] * (
                l_len - len(preds[i])
            )
        )

        for j in range(l_len):
            if (
                j < len(pred_seq)
                and true_seq[j] == pred_seq[j]
            ):
                correct_chars += 1

        total_chars += l_len

    return (
        correct_chars / total_chars
        if total_chars > 0
        else 0.0
    )

def train_crnn_on_4fonts(
    base_dirs,
    num_epochs=30,
    batch_size=16,
    img_h=64,
    patience=5,
    alpha_ctc=0.5
):
    device = torch.device(
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    )

    transform = transforms.Compose([
        transforms.Resize(
            (img_h,256)
        ),
        transforms.ToTensor(),
        transforms.Normalize(
            (0.5,),
            (0.5,)
        )
    ])

    all_images, all_labels = (
        get_all_fonts_image_label_paths(
            base_dirs
        )
    )

    print(
        f"Found {len(all_images)} total images across all fonts (HD + Distorted)"
    )

    dataset = OCRDataset(
        all_images,
        all_labels,
        transform
    )

    split_idx = int(
        0.9 * len(dataset)
    )

    train_ds = torch.utils.data.Subset(
        dataset,
        list(range(split_idx))
    )

    val_ds = torch.utils.data.Subset(
        dataset,
        list(
            range(
                split_idx,
                len(dataset)
            )
        )
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=collate_fn
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate_fn
    )

    n_classes = len(CHAR2IDX)+1

    model = CRNN_STN_Transformer_Attn(
        img_h,
        1,
        n_classes
    ).to(device)

    criterion_ctc = nn.CTCLoss(
        blank=0,
        zero_infinity=True
    )

    criterion_attn = nn.CrossEntropyLoss(
        ignore_index=0
    )

    optimizer = optim.Adam(
        model.parameters(),
        lr=1e-4,
        weight_decay=1e-4
    )

    best_val_loss = float("inf")
    patience_counter = 0

    history = {
        "train_loss":[],
        "val_loss":[],
        "train_acc":[],
        "val_acc":[]
    }

    for epoch in range(
        1,
        num_epochs+1
    ):
        model.train()

        train_loss = 0.0
        total_chars = 0
        correct_chars = 0

        for (
            images,
            labels_flat,
            padded_labels,
            label_lens
        ) in train_loader:

            images = images.to(device)
            labels_flat = labels_flat.to(device)
            padded_labels = padded_labels.to(device)
            label_lens = label_lens.to(device)

            optimizer.zero_grad()

            ctc_logits, attn_logits = model(
                images,
                targets=padded_labels
            )

            log_probs = ctc_logits.permute(
                1,
                0,
                2
            )

            input_lens = torch.full(
                (images.size(0),),
                log_probs.size(0),
                dtype=torch.long,
                device=device
            )

            ctc_loss = criterion_ctc(
                log_probs,
                labels_flat,
                input_lens,
                label_lens
            )

            attn_loss = criterion_attn(
                attn_logits.view(
                    -1,
                    n_classes
                ),
                padded_labels.view(-1)
            )

            loss = (
                alpha_ctc * ctc_loss
                + (1-alpha_ctc) * attn_loss
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                5
            )

            optimizer.step()

            train_loss += loss.item()

            preds = attn_greedy_decoder(
                attn_logits.detach()
            )

            acc = char_accuracy(
                preds,
                padded_labels,
                label_lens
            )

            correct_chars += (
                acc * label_lens.sum().item()
            )

            total_chars += (
                label_lens.sum().item()
            )

        train_loss /= max(
            1,
            len(train_loader)
        )

        train_acc = (
            correct_chars / total_chars
            if total_chars > 0
            else 0.0
        )

        model.eval()

        val_loss = 0.0
        total_chars = 0
        correct_chars = 0

        with torch.no_grad():
            for (
                images,
                labels_flat,
                padded_labels,
                label_lens
            ) in val_loader:

                images = images.to(device)
                labels_flat = labels_flat.to(device)
                padded_labels = padded_labels.to(device)
                label_lens = label_lens.to(device)

                ctc_logits, attn_logits = model(
                    images,
                    targets=padded_labels,
                    teacher_forcing_ratio=0.0
                )

                log_probs = ctc_logits.permute(
                    1,
                    0,
                    2
                )

                input_lens = torch.full(
                    (images.size(0),),
                    log_probs.size(0),
                    dtype=torch.long,
                    device=device
                )

                ctc_loss = criterion_ctc(
                    log_probs,
                    labels_flat,
                    input_lens,
                    label_lens
                )

                attn_loss = criterion_attn(
                    attn_logits.view(
                        -1,
                        n_classes
                    ),
                    padded_labels.view(-1)
                )

                loss = (
                    alpha_ctc * ctc_loss
                    + (1-alpha_ctc) * attn_loss
                )

                val_loss += loss.item()

                preds = attn_greedy_decoder(
                    attn_logits
                )

                acc = char_accuracy(
                    preds,
                    padded_labels,
                    label_lens
                )

                correct_chars += (
                    acc * label_lens.sum().item()
                )

                total_chars += (
                    label_lens.sum().item()
                )

        val_loss /= max(
            1,
            len(val_loader)
        )

        val_acc = (
            correct_chars / total_chars
            if total_chars > 0
            else 0.0
        )

        history["train_loss"].append(
            train_loss
        )

        history["val_loss"].append(
            val_loss
        )

        history["train_acc"].append(
            train_acc
        )

        history["val_acc"].append(
            val_acc
        )

        print(
            f"Epoch {epoch}: "
            f"train_loss={train_loss:.4f}, "
            f"val_loss={val_loss:.4f}, "
            f"train_acc={train_acc:.4f}, "
            f"val_acc={val_acc:.4f}"
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0

            torch.save(
                model.state_dict(),
                "CNN_4fonts_HD_Distorted_best.pth"
            )

            print(
                "Validation loss improved, model saved."
            )

        else:
            patience_counter += 1

            print(
                f"No improvement. "
                f"Patience counter: "
                f"{patience_counter}/{patience}"
            )

            if patience_counter >= patience:
                print(
                    "Early stopping triggered!"
                )
                break

    model.load_state_dict(
        torch.load(
            "CNN_4fonts_HD_Distorted_best.pth"
        )
    )

    print(
        "Restored best model"
    )

    epochs = range(
        1,
        len(history["train_loss"])+1
    )

    plt.figure(
        figsize=(12,5)
    )

    plt.subplot(
        1,
        2,
        1
    )

    plt.plot(
        epochs,
        history["train_loss"],
        label="Train Loss"
    )

    plt.plot(
        epochs,
        history["val_loss"],
        label="Val Loss"
    )

    plt.xlabel("Epochs")
    plt.ylabel("Loss")
    plt.title("Loss Curve")
    plt.legend()

    plt.subplot(
        1,
        2,
        2
    )

    plt.plot(
        epochs,
        history["train_acc"],
        label="Train Acc"
    )

    plt.plot(
        epochs,
        history["val_acc"],
        label="Val Acc"
    )

    plt.xlabel("Epochs")
    plt.ylabel("Char Accuracy")
    plt.title("Character-level Accuracy")
    plt.legend()

    plt.tight_layout()
    plt.show()

if __name__=="__main__":
    base_dirs = [
        "/content/All_4_Fonts_Dataset_Distorted",
        "/content/All_4_Fonts_Dataset_HD"
    ]

    train_crnn_on_4fonts(
        base_dirs,
        num_epochs=30,
        batch_size=16,
        img_h=64,
        patience=5
    )

🔹 Found 125076 total images across all fonts (HD + Distorted)
Epoch 1: train_loss=1.4030, val_loss=0.8156, train_acc=0.3723, val_acc=0.5055
✅ Validation loss improved, model saved.
Epoch 2: train_loss=0.5693, val_loss=0.4568, train_acc=0.6861, val_acc=0.7357
✅ Validation loss improved, model saved.
Epoch 3: train_loss=0.3565, val_loss=0.3397, train_acc=0.8157, val_acc=0.8102
✅ Validation loss improved, model saved.
Epoch 4: train_loss=0.2637, val_loss=0.2802, train_acc=0.8702, val_acc=0.8513
✅ Validation loss improved, model saved.
Epoch 5: train_loss=0.2126, val_loss=0.2193, train_acc=0.8994, val_acc=0.8904
✅ Validation loss improved, model saved.
Epoch 6: train_loss=0.1813, val_loss=0.1835, train_acc=0.9173, val_acc=0.9112
✅ Validation loss improved, model saved.
Epoch 7: train_loss=0.1607, val_loss=0.2300, train_acc=0.9282, val_acc=0.8810
⚠️ No improvement. Patience counter: 1/5
Epoch 8: train_loss=0.1453, val_loss=0.1535, train_acc=0.9362, val_acc=0.9264
✅ Validation loss improved,

In [2]:
import os
import glob
import torch
import math
from PIL import Image
from jiwer import wer, cer
from torchvision import transforms
import torch.nn as nn
import torch.nn.functional as F

charset_file = "/content/kashmiri_charset (2).txt"

with open(charset_file, encoding="utf-8") as f:
    CHARS = f.read().strip()

CHAR2IDX = {c: i + 1 for i, c in enumerate(CHARS)}
IDX2CHAR = {i: c for c, i in CHAR2IDX.items()}

def labels_to_text(labels):
    return ''.join([IDX2CHAR.get(int(i), '') for i in labels])

class STN(nn.Module):
    def __init__(self, img_h, img_w):
        super().__init__()
        self.localization = nn.Sequential(
            nn.Conv2d(1, 8, 7, padding=3),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(8, 16, 5, padding=2),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        fc_input = 16 * (img_h // 4) * (img_w // 4)

        self.fc_loc = nn.Sequential(
            nn.Linear(fc_input, 32),
            nn.ReLU(),
            nn.Linear(32, 6)
        )

        self.fc_loc[2].weight.data.zero_()
        self.fc_loc[2].bias.data.copy_(
            torch.tensor([1, 0, 0, 0, 1, 0], dtype=torch.float)
        )

    def forward(self, x):
        xs = self.localization(x)
        xs = xs.view(xs.size(0), -1)
        theta = self.fc_loc(xs).view(-1, 2, 3)
        grid = F.affine_grid(theta, x.size(), align_corners=False)
        x = F.grid_sample(x, grid, align_corners=False)
        return x

class ResidualBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, stride, 1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(),
            nn.Conv2d(out_ch, out_ch, 3, 1, 1),
            nn.BatchNorm2d(out_ch)
        )

        self.relu = nn.ReLU()

        self.downsample = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 1, stride),
            nn.BatchNorm2d(out_ch)
        ) if in_ch != out_ch or stride != 1 else nn.Identity()

    def forward(self, x):
        out = self.conv(x)
        out += self.downsample(x)
        return self.relu(out)

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()

        div_term = torch.exp(
            torch.arange(0, d_model, 2).float()
            * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)

        self.register_buffer("pe", pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

class SmallTransformerEncoder(nn.Module):
    def __init__(self, feature_dim, nhead=4, num_layers=2, dropout=0.1):
        super().__init__()

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=feature_dim,
            nhead=nhead,
            dim_feedforward=feature_dim * 2,
            dropout=dropout,
            batch_first=True
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        self.pos_encoder = PositionalEncoding(feature_dim)

    def forward(self, x):
        x = self.pos_encoder(x)
        return self.encoder(x)

class AttentionDecoder(nn.Module):
    def __init__(self, feature_dim, hidden_dim, n_classes, dropout=0.1):
        super().__init__()

        self.hidden_dim = hidden_dim

        self.embedding = nn.Embedding(
            n_classes,
            hidden_dim
        )

        self.lstm = nn.LSTM(
            hidden_dim + feature_dim,
            hidden_dim,
            batch_first=True
        )

        self.attn_fc = nn.Linear(
            hidden_dim + feature_dim,
            1
        )

        self.out_fc = nn.Linear(
            hidden_dim,
            n_classes
        )

        self.dropout = nn.Dropout(dropout)

    def forward(
        self,
        encoder_outputs,
        targets=None,
        teacher_forcing_ratio=0.5
    ):
        batch_size, seq_len, feature_dim = encoder_outputs.size()
        device = encoder_outputs.device

        max_len = (
            targets.size(1)
            if targets is not None
            else seq_len
        )

        h = torch.zeros(
            1,
            batch_size,
            self.hidden_dim,
            device=device
        )

        c = torch.zeros(
            1,
            batch_size,
            self.hidden_dim,
            device=device
        )

        inputs = torch.zeros(
            batch_size,
            1,
            self.hidden_dim,
            device=device
        )

        outputs = []

        for t in range(max_len):

            attn_weights = F.softmax(
                self.attn_fc(
                    torch.cat(
                        [
                            inputs.squeeze(1)
                            .unsqueeze(1)
                            .repeat(1, seq_len, 1),
                            encoder_outputs
                        ],
                        dim=2
                    )
                ),
                dim=1
            )

            context = torch.sum(
                attn_weights * encoder_outputs,
                dim=1,
                keepdim=True
            )

            lstm_input = torch.cat(
                [inputs, context],
                dim=2
            )

            out, (h, c) = self.lstm(
                lstm_input,
                (h, c)
            )

            out = self.dropout(out)

            pred = self.out_fc(out)

            outputs.append(pred)

            if (
                targets is not None
                and torch.rand(1).item()
                < teacher_forcing_ratio
            ):
                idx = targets[:, t].unsqueeze(1)
                inputs = self.embedding(idx)
            else:
                idx = pred.argmax(-1)
                inputs = self.embedding(idx)

        outputs = torch.cat(outputs, dim=1)

        return outputs

class CRNN_STN_Transformer_Attn(nn.Module):
    def __init__(
        self,
        img_h=64,
        n_channels=1,
        n_classes=len(CHAR2IDX) + 1,
        dropout=0.2,
        hidden_dim=256
    ):
        super().__init__()

        self.n_classes = n_classes

        self.stn = STN(img_h, 256)

        self.cnn = nn.Sequential(
            ResidualBlock(n_channels, 64),
            nn.MaxPool2d(2, 2),
            ResidualBlock(64, 128),
            nn.MaxPool2d(2, 2),
            ResidualBlock(128, 256),
            nn.MaxPool2d((2, 1)),
            ResidualBlock(256, 256),
            nn.MaxPool2d((2, 1)),
            nn.Dropout(dropout)
        )

        self.h_out = img_h // 16
        feature_dim = 256 * self.h_out

        self.transformer = SmallTransformerEncoder(
            feature_dim,
            nhead=4,
            num_layers=2,
            dropout=0.1
        )

        self.fc_ctc = nn.Linear(
            feature_dim,
            n_classes
        )

        self.attn_decoder = AttentionDecoder(
            feature_dim,
            hidden_dim,
            n_classes,
            dropout
        )

    def forward(
        self,
        x,
        targets=None,
        teacher_forcing_ratio=0.5
    ):
        x = self.stn(x)
        x = self.cnn(x)

        b, c, h, w = x.size()

        x = (
            x.permute(0, 3, 1, 2)
            .contiguous()
            .view(b, w, c * h)
        )

        enc_out = self.transformer(x)

        ctc_out = self.fc_ctc(
            enc_out
        ).log_softmax(2)

        attn_out = self.attn_decoder(
            enc_out,
            targets,
            teacher_forcing_ratio
        )

        return ctc_out, attn_out

def ctc_greedy_decoder(ctc_out, blank=0):
    preds = ctc_out.argmax(-1)

    pred_texts_idx = []

    for b in range(preds.size(1)):
        seq = []
        prev = blank

        for t in range(preds.size(0)):
            p = int(preds[t, b].item())

            if p != prev and p != blank:
                seq.append(p)

            prev = p

        pred_texts_idx.append(seq)

    return pred_texts_idx

def evaluate_folder(model, test_folder, transform):
    model.eval()

    device = next(model.parameters()).device

    image_paths = sorted(
        glob.glob(
            os.path.join(
                test_folder,
                "**",
                "*.png"
            ),
            recursive=True
        )
    )

    total_wer = 0.0
    total_cer = 0.0
    n = 0

    with torch.no_grad():

        for img_path in image_paths:

            txt_path = img_path.replace(
                ".png",
                ".txt"
            )

            if not os.path.exists(txt_path):
                continue

            try:
                image = Image.open(
                    img_path
                ).convert("L")
            except:
                continue

            with open(
                txt_path,
                encoding="utf-8"
            ) as f:
                gt_text = f.read().strip()

            img_tensor = transform(
                image
            ).unsqueeze(0).to(device)

            ctc_out, attn_out = model(
                img_tensor,
                targets=None,
                teacher_forcing_ratio=0.0
            )

            ctc_out = ctc_out.permute(
                1,
                0,
                2
            )

            pred_idx = ctc_greedy_decoder(
                ctc_out,
                blank=0
            )[0]

            pred_text = labels_to_text(
                pred_idx
            )

            total_wer += wer(
                gt_text,
                pred_text
            )

            total_cer += cer(
                gt_text,
                pred_text
            )

            n += 1

    avg_wer = total_wer / max(1, n)
    avg_cer = total_cer / max(1, n)

    return avg_wer, avg_cer, n

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

transform = transforms.Compose([
    transforms.Resize((64, 256)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

n_classes = len(CHAR2IDX) + 1

model = CRNN_STN_Transformer_Attn(
    img_h=64,
    n_channels=1,
    n_classes=n_classes
).to(device)

ckpt_path = "/content/CNN_4fonts_HD_Distorted_best (4).pth"

if not os.path.exists(ckpt_path):
    raise FileNotFoundError(
        "Checkpoint not found: " + ckpt_path
    )

model.load_state_dict(
    torch.load(
        ckpt_path,
        map_location=device
    )
)

model.eval()

fonts = [
    "NotoNastaliqUrdu-Medium",
    "Gulmarg Nataleeq_2013",
    "Narqalam",
    "NaskhArabic"
]

base_hd = "/content/All_4_Fonts_Dataset_HD"
base_dist = "/content/All_4_Fonts_Dataset_Distorted"

results = []

for font in fonts:

    for dataset_name, base_dir in [
        ("HD", base_hd),
        ("Distorted", base_dist)
    ]:

        test_dir = os.path.join(
            base_dir,
            font,
            "test"
        )

        if not os.path.exists(test_dir):
            continue

        avg_wer, avg_cer, n = evaluate_folder(
            model,
            test_dir,
            transform
        )

        results.append(
            (
                dataset_name,
                font,
                avg_wer,
                avg_cer,
                n
            )
        )

if results:

    total_wer = sum(
        r[2] * r[4] for r in results
    )

    total_cer = sum(
        r[3] * r[4] for r in results
    )

    total_images = sum(
        r[4] for r in results
    )

    overall_wer = total_wer / total_images
    overall_cer = total_cer / total_images

    print("Evaluation completed")
    print(f"WER: {overall_wer:.4f}")
    print(f"CER: {overall_cer:.4f}")

Evaluation completed
CER: 0.0101
WER: 0.0454
